<a href="https://colab.research.google.com/github/atilimai/plant-ai-project/blob/main/notebooks/Boran_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Kaggle API setup
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and unzip the dataset
!kaggle datasets download -d emmarex/plantdisease -p data/raw --unzip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:07<00:00, 93.2MB/s]



In [2]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Scan folders and collect image data
all_images = []
base_path = 'data/raw/plantvillage/PlantVillage'

for root, dirs, files in os.walk(base_path):
    for file in files:
        if "___" in file and file.lower().endswith(('.jpg', '.jpeg', '.png')):
            leaf_id = file.split('___')[0]
            label = os.path.basename(root)
            all_images.append({
                'path': os.path.join(root, file),
                'leaf_id': leaf_id,
                'label': label
            })

df = pd.DataFrame(all_images)

# Leaf-ID based non-leaking split (GroupShuffleSplit)
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['leaf_id']))
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

# Leakage Guard Check
intersection = set(train_df['leaf_id']).intersection(set(test_df['leaf_id']))

print(f"✅ Data processing complete.")
print(f"Total Images: {len(df)}")
print(f"Train Set: {len(train_df)} | Test Set: {len(test_df)}")
print(f"🛡️ Leakage Test Result: {len(intersection)} (0 means success)")

✅ Data processing complete.
Total Images: 20638
Train Set: 16510 | Test Set: 4128
🛡️ Leakage Test Result: 0 (0 means success)
